# Advanced tutorial: Build a multivariable dataset

## Research question

**How are daily streamflow, water temperature, dissolved oxygen, and precipitation related at selected STREAM stations?**

## `streamwq` functions used

- `stream.get_sites()`
- `stream.get_data("water_quality", ...)`
- `stream.get_data("streamflow", ...)`
- `stream.get_data("meteorology", ...)`
- `common_observation_period()`
- `resample_timeseries()`
- `join_timeseries()`

In [ ]:
import streamwq as stream
from streamwq.joins import join_timeseries
from streamwq.helpers.pandas import common_observation_period, resample_timeseries

## 1. Select stations with the required observations

In [ ]:
sites = stream.get_sites(
    variables=["WTemp_C", "DO_mgL"],
    streamflow=True,
)
site_ids = sites.slice(0, 3)
stream.get_metadata(site_ids=site_ids, columns=["STREAM_ID", "site_name", "state_name"])

## 2. Retrieve the three time-series datasets

In [ ]:
water_quality = stream.get_data(
    "water_quality",
    site_ids=site_ids,
    columns=["STREAM_ID", "DateTime", "WTemp_C", "DO_mgL"],
    quality="approved",
).to_pandas()
streamflow = stream.get_data(
    "streamflow",
    site_ids=site_ids,
    columns=["STREAM_ID", "DateTime", "Q_m3s"],
    quality="approved",
).to_pandas()
meteorology = stream.get_data(
    "meteorology",
    site_ids=site_ids,
    columns=["STREAM_ID", "DateTime", "APCP_surface"],
).to_pandas()
for dataset_name, table in [
    ("water quality", water_quality),
    ("streamflow", streamflow),
    ("meteorology", meteorology),
]:
    if table.empty:
        raise ValueError(f"No {dataset_name} rows were returned for the selected sites")
common_start, common_end = common_observation_period(
    water_quality, streamflow, meteorology
)
print(common_start, common_end)

## 3. Aggregate to a common daily resolution

Temperature, oxygen, and flow use daily means. Precipitation uses a daily sum.

In [ ]:
daily_wq = resample_timeseries(
    water_quality, ["WTemp_C", "DO_mgL"], "daily", "mean"
)
daily_flow = resample_timeseries(streamflow, "Q_m3s", "daily", "mean")
daily_rain = resample_timeseries(meteorology, "APCP_surface", "daily", "sum")

## 4. Join exact daily station/date records

Resampling first gives every dataset the same daily timestamps, allowing the package's exact and validated join to be used.

In [ ]:
analysis_data = join_timeseries(daily_wq, daily_flow)
analysis_data = join_timeseries(analysis_data, daily_rain)
print(f"Analysis-ready rows: {len(analysis_data):,}")
analysis_data.head()

In [ ]:
analysis_data[["WTemp_C", "DO_mgL", "Q_m3s", "APCP_surface"]].corr()